In [3]:
%load_ext autoreload
%autoreload 2

import sys
import os
# Thêm thư mục cha (thư mục gốc của project) vào đường dẫn hệ thống
sys.path.append(os.path.abspath('..'))

import torch
import torch.nn as nn
import numpy as np
from pathlib import Path

# Import từ thư mục core
from core.utils import seed_everything, get_device, build_dataloaders, train_one_epoch, evaluate, count_parameters, plot_history
from core.models import CNN

# Cấu hình Hyperparameters (Thay thế cho argparse)
EPOCHS = 10
BATCH_SIZE = 32
LR = 1e-3
VAL_RATIO = 0.1
SEED = 42
OUT_DIR = Path("runs_mnist_cnn_baseline")
OUT_DIR.mkdir(parents=True, exist_ok=True)

seed_everything(SEED)
device = get_device()
print(f"Sử dụng thiết bị: {device}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Sử dụng thiết bị: cpu


In [4]:
train_loader, val_loader, test_loader = build_dataloaders(BATCH_SIZE, VAL_RATIO, SEED)

model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(f"Tổng số tham số: {count_parameters(model):,}")

Failed to read module file 'd:\Repos\BSNN_EdgeAI_Workspace\core\utils.py' for module 'core.utils': UnicodeDecodeError
Traceback (most recent call last):
  File "d:\Repos\BSNN_EdgeAI_Workspace\env_tf\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 219, in update_sources
    self.source_by_modname[new_modname] = f.read()
                                          ~~~~~~^^
  File "C:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 866: character maps to <undefined>
100.0%
100.0%
100.0%
100.0%

Tổng số tham số: 54,410


In [5]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}

# Lưu mô hình tốt nhất
if best_state is not None:
    torch.save(best_state, OUT_DIR / "best_model.pt")
    model.load_state_dict(best_state)

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print("-" * 40)
print(f"Test accuracy: {test_acc*100:.2f}%")

Epoch 01/10 | Train Acc: 93.86% | Val Acc: 96.73%
Epoch 02/10 | Train Acc: 97.72% | Val Acc: 97.87%
Epoch 03/10 | Train Acc: 98.33% | Val Acc: 97.82%
Epoch 04/10 | Train Acc: 98.64% | Val Acc: 98.15%
Epoch 05/10 | Train Acc: 98.85% | Val Acc: 98.05%
Epoch 06/10 | Train Acc: 99.01% | Val Acc: 98.17%
Epoch 07/10 | Train Acc: 99.21% | Val Acc: 98.15%
Epoch 08/10 | Train Acc: 99.26% | Val Acc: 98.17%
Epoch 09/10 | Train Acc: 99.46% | Val Acc: 98.25%
Epoch 10/10 | Train Acc: 99.52% | Val Acc: 97.95%
----------------------------------------
Test accuracy: 98.10%


In [6]:
# Gọi hàm vẽ biểu đồ
plot_history(history, OUT_DIR)

# Lưu log numpy
np.save(OUT_DIR / "train_loss.npy", np.array(history["train_loss"]))
np.save(OUT_DIR / "train_acc.npy", np.array(history["train_acc"]))
np.save(OUT_DIR / "val_loss.npy", np.array(history["val_loss"]))
np.save(OUT_DIR / "val_acc.npy", np.array(history["val_acc"]))

print("Đã lưu lịch sử huấn luyện và biểu đồ thành công!")

Đã lưu lịch sử huấn luyện và biểu đồ thành công!
